In [ ]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, random_split

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from scipy.stats import pearsonr

import matplotlib.pyplot as plt 

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
raw_data = "../data/deltas_vs_rmsd.csv"
data = np.loadtxt(raw_data, delimiter=',', skiprows=1)
data = data[1:-1:10]
X = data[:, :3]
y = data[:, 3:]
data.shape

In [ ]:
raw_data = "../data/vtors_vs_rmsd.csv"

data = np.loadtxt(raw_data, delimiter=',', skiprows=1)
data = data[1:-1:10]
X = data[:, :6]
y = data[:, 6:]


In [ ]:
# Normalize features
scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X)

scaler_y = StandardScaler()
y_scaled = scaler_y.fit_transform(y)

# Convert to PyTorch tensors
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_tensor = torch.tensor(y_scaled, dtype=torch.float32)

full_dataset = TensorDataset(X_tensor, y_tensor)

# Split into training and validation sets (80% train, 20% val)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128)


In [ ]:
# Define the model
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(6, 32)
        self.fc2 = nn.Linear(32, 16)
        self.fc3 = nn.Linear(16, 8)
        self.fc4 = nn.Linear(8, 1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.dropout(self.relu(self.fc2(x)))
        x = self.dropout(self.relu(self.fc3(x)))
        x = self.fc4(x)
        return x


# Initialize model, loss function, optimizer
model = Net().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)



In [ ]:
# Training with validation
epochs = 50
training_loss = []
validation_loss = []
for epoch in range(epochs):
    # Training phase
    model.train()
    train_loss = 0.0
    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # Validation phase
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for val_x, val_y in val_loader:
            val_x = val_x.to(device)
            val_y = val_y.to(device)
            val_outputs = model(val_x)
            v_loss = criterion(val_outputs, val_y)
            val_loss += v_loss.item()

    # Averages
    train_loss /= len(train_loader)
    val_loss /= len(val_loader)

    training_loss.append(train_loss)
    validation_loss.append(val_loss)
    
    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")


In [ ]:
plt.style.use('ggplot')
plt.plot(training_loss, label='Training Loss')
plt.plot(validation_loss, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()


In [ ]:
# Test the model
model.eval()
with torch.no_grad():
    test_preds = model(val_dataset.dataset.tensors[0])
    test_preds_rescaled = scaler_y.inverse_transform(test_preds.numpy())
    y_test_rescaled = scaler_y.inverse_transform(y_test.numpy())

plt.style.use('ggplot')
plt.scatter(y_test_rescaled, test_preds_rescaled, marker='o', s=1, color='blue', alpha=0.1)
plt.xlabel('RMSD K-U algorithm')
plt.ylabel('NN Predictions')

print("RMSE ", mean_squared_error(y_test_rescaled[:,0], test_preds_rescaled[:,0]))
print(pearsonr(y_test_rescaled[:,0], test_preds_rescaled[:,0]))